# Small-schema MCP evaluation

Run `RDFSOLVE_NOTEBOOK=01_small.ipynb sbatch scripts/slurm_qwen_mcp.sh` after `00_mine.ipynb`. Each model sees the question and saved schema. Reference queries and complete answer bindings stay in this notebook. Precision and recall compare exact RDF tuples on the fixed snapshots.


In [ ]:
from pathlib import Path
import os, json
from rdfsolve.api import ask_rdf
from rdfsolve.client.api import Client

folder = Path(os.environ['RDFSOLVE_ROOT']) / 'notebooks/mcp/schemas'
output = Path(os.environ['RDFSOLVE_OUTPUT'])
aop = 'http://aopkb.org/aop_ontology#'
wp = 'http://vocabularies.wikipathways.org/wp#'
cases = [
    ('local-study', 'Return every Researcher and the year of each of their employment records, when available. Keep researchers without employment records. Use output variables researcher and year.',
     ['researcher','year'], 'SELECT DISTINCT ?researcher ?year WHERE { ?researcher a <urn:study:Researcher> . OPTIONAL { ?researcher <urn:study:employment> ?job . ?job a <urn:study:Employment> . OPTIONAL { ?job <urn:study:year> ?year } } }', 'urn:study:Researcher', 'urn:study:Employment'),
    ('aopwikirdf-small', 'List all Adverse Outcome Pathway resource identities. Use output variable pathway.',
     ['pathway'], f'SELECT DISTINCT ?pathway WHERE {{ ?pathway a <{aop}AdverseOutcomePathway> }}', aop+'AdverseOutcomePathway', None),
    ('aopwikirdf-small', 'Return each Adverse Outcome Pathway and its Key Events linked by has key event, when available. Keep pathways with no events. Use output variables pathway and event.',
     ['pathway','event'], f'SELECT DISTINCT ?pathway ?event WHERE {{ ?pathway a <{aop}AdverseOutcomePathway> . OPTIONAL {{ ?pathway <{aop}has_key_event> ?event . ?event a <{aop}KeyEvent> }} }}', aop+'AdverseOutcomePathway', aop+'KeyEvent'),
    ('wikipathways-small', 'List all Pathway resource identities. Use output variable pathway.',
     ['pathway'], f'SELECT DISTINCT ?pathway WHERE {{ ?pathway a <{wp}Pathway> }}', wp+'Pathway', None),
    ('wikipathways-small', 'List every Pathway with its available title. Keep pathways without titles. Use output variables pathway and title.',
     ['pathway','title'], f'SELECT DISTINCT ?pathway ?title WHERE {{ ?pathway a <{wp}Pathway> . OPTIONAL {{ ?pathway <http://purl.org/dc/elements/1.1/title> ?title }} }}', wp+'Pathway', None),
]

def terms(rows, columns):
    return {tuple(json.dumps(row.get(c),sort_keys=True,ensure_ascii=False) for c in columns) for row in rows}


In [ ]:
metrics = []
for index, (source, question, columns, reference, start, target) in enumerate(cases):
    schema, data = folder / f'{source}.schema.json', folder / f'{source}.ttl'
    with Client.open(schema, data_file=data, graph_uris=[]) as client:
        expected = client.select(reference)
        gold = [{k: {'type':v.type, 'value':v.value, **({'xml:lang':v.lang} if v.lang else {}), **({'datatype':v.datatype} if v.datatype else {})} for k,v in row.items()} for row in expected.rows]
        difficulty = None
        if target:
            routes = client.paths_between(start, target, max_hops=3, max_paths=100, allow_partial=True)
            difficulty = min((len(r) for r in routes.attrs['routes']), default=None)
    print('QUESTION:', question)
    answer = await ask_rdf(question, schema=schema, data_file=data, graph_uris=[], source_id=source)
    expected_rows, predicted_rows = terms(gold, columns), terms(answer.bindings, columns)
    overlap = len(expected_rows & predicted_rows)
    precision = overlap / len(predicted_rows) if predicted_rows else 0
    recall = overlap / len(expected_rows) if expected_rows else float(not predicted_rows)
    metric = dict(source=source, question=question, state=answer.state, reference_rows=len(expected_rows), predicted_rows=len(predicted_rows), precision=precision, recall=recall, f1=2*precision*recall/(precision+recall) if precision+recall else 0, shortest_mined_path=difficulty, diagnostics=answer.diagnostics())
    metrics.append(metric)
    (output / 'small-metrics.json').write_text(json.dumps(metrics,indent=2))
    (output / f'reference-{index}.json').write_text(json.dumps(dict(query=reference,bindings=gold),indent=2))
    print(answer.text)
    print({k:v for k,v in metric.items() if k!='diagnostics'})
    print(answer.query)


In [ ]:
assert all(m['state']=='complete' and m['f1']==1 for m in metrics), 'See small-metrics.json and the saved call journals for failures.'